# Field F1 from raw extractions — standalone

Reads a **`raw_extractions.jsonl`** (one record per image, holding the model's `raw_response`)
plus a **ground-truth file**, parses and scores them *in this notebook*, and reports:

1. Two per-document headline aggregates and their spread.
2. Predicted vs actual `DOCUMENT_TYPE`, and a reconciliation of which fields are scored.
3. **F1 per extraction field** by document type, as a macro table and a micro table.
4. Optionally, a **paired clean-vs-degraded comparison** of two runs over the same documents.

## This notebook does not reproduce the pipeline's numbers

It carries **its own scorer**. The production chain (`stages/clean` → `stages/evaluate`) is
~3,500 lines across five modules; what is here is a compact reimplementation, so **its figures
will differ from a run's reported F1 — from the first cell, not merely as versions drift.**

Deliberately not implemented: bank balance math enhancement, GST consistency validation, ABN
checksum matching, truncated-JSON repair, debit-only filtering. All are LMM_POC-specific and
irrelevant to scoring a different model.

Quote these numbers as *this notebook's* F1, never as the pipeline's.

## What it needs

Nothing but `pandas`, `matplotlib` and the standard library — **no repo checkout, no imports from
the codebase, no YAML, no environment variables.** Every setting is an explicit literal in the
config cell below. Point it at your own model's output and your own ground truth, edit
`FIELDS_ASKED_BY_DOC_TYPE` and `FIELD_TYPES`, and it runs.

The declared field sets mirror `config/extraction_schema.yml`, so if that contract changes, the
config cell must change with it. The ground truth itself is whatever your run was scored against;
for this corpus, prefer the `ground_truth.csv` that `python -m generators.pipeline eval-set`
writes beside the images, and see the config cell for how it differs from the older
`scripts/generate_extraction_gt.py` output.

## Which average?

The three numbers here are all "mean F1" and all differ. They are not competing estimates of one
quantity; they answer different questions:

| | unit of account | computed as |
|---|---|---|
| headline summary | one document | mean over documents of a per-document field aggregate |
| macro field table | one document | mean over documents of that field's F1 |
| micro field table | one extracted item | F1 of the pooled tp/fp/fn for that field |

For single-valued fields macro and micro diverge from the list-valued ones, because micro lets a
31-transaction statement outweigh a one-line receipt 31 to 1. Always say which one you are
quoting.

The comparison section at the end adds a fourth: a **paired difference**, whose unit of account is
one document scored twice. It is not comparable to any of the above — see that section.

In [ ]:
from pathlib import Path

# ===========================================================================
# INPUTS
#
# GT_PATH accepts either ground-truth file this repo produces. They carry the
# same columns — both project config/extraction_schema.yml — but differ in two
# ways that show up in this notebook's output:
#
#   `python -m generators.pipeline eval-set`  ->  ground_truth.csv / .jsonl
#       PREFER THIS. Written into both the clean and the degraded output
#       directory, as byte-identical copies. Filenames are generic
#       (CASE001_receipt.png), so the layout variant cannot leak to the model,
#       and fields a type is not asked for are masked to NOT_FOUND.
#
#   `python scripts/generate_extraction_gt.py --output ...`
#       The older standalone script. Filenames embed the layout
#       (CASE001_woolworths_standard.png), and any field the ground truth
#       happens to carry is populated even when no prompt asks for it — so the
#       reconciliation cell below reports bank SUPPLIER_NAME / PAYER_NAME as
#       "real GT, never asked". That is the file talking, not the model.
#
# The .csv and .jsonl of a given export are interchangeable and score identically,
# including on misclassified documents. They are not the same shape — the CSV pads
# every row out to the union of all types' columns with NOT_FOUND, while the JSONL
# is ragged, each record carrying only its own type's fields — so the scored field
# set is taken from the file as a whole rather than from one record. See
# ground_truth_columns below for what goes wrong if it is not.
# ===========================================================================
RAW_PATH = Path("/path/to/raw_extractions.jsonl")
GT_PATH = Path("/path/to/ground_truth.csv")  # .csv (wide) or .jsonl (per-type)

# Second run to compare against, scored on the SAME ground truth — normally the
# degraded half of an eval-set export against RAW_PATH's clean half. The two
# directories hold identical filenames and identical ground truth by
# construction, so image quality is the only variable between them.
#
# None disables the comparison section at the end of the notebook. That is an
# explicit choice, not a default: with one run there is nothing to difference.
RAW_PATH_B = None
RUN_LABELS = ("clean", "degraded")  # (RAW_PATH, RAW_PATH_B), used in table headers

# ===========================================================================
# FIELD SETS — what the model was ASKED for, per document type.
#
# This is the authority for what gets scored, and it must be DECLARED rather
# than inferred:
#   * inferring it from which ground-truth cells are populated drops fields
#     whose correct answer is NOT_FOUND (see NOT_FOUND note below);
#   * inferring it from the keys the model emitted admits fields the model
#     invented and nobody asked for.
# Mirrors config/extraction_schema.yml, which is itself the contract the
# extraction prompt must match. Invoice and receipt are field-identical there,
# which is why a receipt misclassified as an invoice is still asked the right
# questions and loses only DOCUMENT_TYPE.
# ===========================================================================
_INVOICE_FIELDS = [
    "DOCUMENT_TYPE",
    "BUSINESS_ABN",
    "SUPPLIER_NAME",
    "BUSINESS_ADDRESS",
    "PAYER_NAME",
    "PAYER_ADDRESS",
    "INVOICE_DATE",
    "LINE_ITEM_DESCRIPTIONS",
    "LINE_ITEM_QUANTITIES",
    "LINE_ITEM_PRICES",
    "LINE_ITEM_TOTAL_PRICES",
    "IS_GST_INCLUDED",
    "GST_AMOUNT",
    "TOTAL_AMOUNT",
]

FIELDS_ASKED_BY_DOC_TYPE = {
    "BANK_STATEMENT": [
        "DOCUMENT_TYPE",
        "STATEMENT_DATE_RANGE",
        "LINE_ITEM_DESCRIPTIONS",
        "TRANSACTION_DATES",
        "TRANSACTION_AMOUNTS_PAID",
    ],
    "INVOICE": list(_INVOICE_FIELDS),
    "RECEIPT": list(_INVOICE_FIELDS),
}

# Retired fields. Never scored, even if a ground-truth file carries a column
# for one — a GT column here is a defect in the GT file, not a signal to score.
LEGACY_FIELDS = {"ACCOUNT_BALANCE"}

# ===========================================================================
# COMPARISON TYPES — how each field's value is matched.
#   text / id / date / date_range / monetary / boolean
#   *_list variants are " | "-separated and scored position-wise
# ===========================================================================
FIELD_TYPES = {
    "DOCUMENT_TYPE": "exact",
    "BUSINESS_ABN": "id",
    "SUPPLIER_NAME": "text",
    "BUSINESS_ADDRESS": "text",
    "PAYER_NAME": "text",
    "PAYER_ADDRESS": "text",
    "INVOICE_DATE": "date",
    "STATEMENT_DATE_RANGE": "date_range",
    "LINE_ITEM_DESCRIPTIONS": "text_list",
    "LINE_ITEM_QUANTITIES": "numeric_list",
    "LINE_ITEM_PRICES": "monetary_list",
    "LINE_ITEM_TOTAL_PRICES": "monetary_list",
    "IS_GST_INCLUDED": "boolean",
    "GST_AMOUNT": "monetary",
    "TOTAL_AMOUNT": "monetary",
    "TRANSACTION_DATES": "date_list",
    "TRANSACTION_AMOUNTS_PAID": "monetary_list",
}
DEFAULT_FIELD_TYPE = "text"

# ===========================================================================
# MATCHING KNOBS
# ===========================================================================
LIST_SEPARATOR = "|"
NOT_FOUND = "NOT_FOUND"
FUZZY_THRESHOLD = 0.85  # SequenceMatcher ratio above which two texts match
MONETARY_TOLERANCE = 0.01  # absolute currency difference treated as equal
DATE_FORMATS = [
    "%d/%m/%Y",
    "%d-%m-%Y",
    "%Y-%m-%d",
    "%d %b %Y",
    "%d %B %Y",
    "%d/%m/%y",
]

# NOT_FOUND is a FIRST-CLASS EXPECTED ANSWER, not a marker to filter on.
# Where the ground truth says NOT_FOUND, emitting NOT_FOUND is correct and
# emitting a value is a false positive. That is what catches a model putting
# the supplier's name into PAYER_NAME on a receipt that shows no payer.

# ===========================================================================
# GROUND-TRUTH ROW FILTERS — drop GT rows the prompt never asks for.
#
# Bank statement GT lists EVERY transaction, with TRANSACTION_AMOUNTS_PAID set
# to NOT_FOUND on credits, whereas the prompt asks for debits only. Those credit
# rows are not misses: they were never requested. Left in, each one shifts every
# later position by one and position-wise scoring collapses — on the reference
# file this drags bank F1 from a real result down to ~0.15.
#
# Still load-bearing on the current corpus: 266 of its 1489 bank transactions
# are credits. Each entry drops positions where `require_present` is NOT_FOUND,
# applying the same mask to every parallel field. Set to {} if your GT lists
# only what the prompt asks for.
# ===========================================================================
GT_ROW_FILTERS = {
    "BANK_STATEMENT": {
        "require_present": "TRANSACTION_AMOUNTS_PAID",
        "fields": [
            "TRANSACTION_DATES",
            "TRANSACTION_AMOUNTS_PAID",
            "LINE_ITEM_DESCRIPTIONS",
        ],
    },
}

# ===========================================================================
# ORDER NORMALISATION — mirrors the pipeline's pre-scoring chronological sort.
# Parallel transaction lists are re-ordered by the sort_key field's dates on
# BOTH sides before position-wise scoring, so document order vs chronological
# order is not charged as an extraction error.
# ===========================================================================
TRANSACTION_SORT_GROUPS = {
    "BANK_STATEMENT": {
        "sort_key": "TRANSACTION_DATES",
        "fields": [
            "TRANSACTION_DATES",
            "TRANSACTION_AMOUNTS_PAID",
            "LINE_ITEM_DESCRIPTIONS",
        ],
    },
}

In [ ]:
import json
import re
from difflib import SequenceMatcher
from datetime import datetime

import pandas as pd

_KEY_LINE = re.compile(r"^([A-Z][A-Z0-9_]{2,})\s*:\s*(.*)$")
_BULLET = re.compile(r"^\s*[-*+•]+\s*")


def clean_line(line: str) -> str:
    """Strip markdown bullets and emphasis so a ``KEY: value`` line can match.

    Models wrap field lines in list markers and bold, e.g.
    ``*   **SUPPLIER_NAME:** Acme Pty Ltd``. Removing both leaves a plain
    ``KEY: value`` for the matcher.

    Args:
        line: One line of the raw response.

    Returns:
        The line with bullets and asterisk emphasis removed.
    """
    return _BULLET.sub("", line).replace("**", "").replace("*", "").strip()


def match_key_line(line: str):
    """Match a cleaned line as ``KEY: value``, or return None."""
    return _KEY_LINE.match(clean_line(line))


def _fail(what, where, example, remedy):
    """Raise a diagnostic error carrying what/where/example/remedy."""
    raise ValueError(
        f"{what}\n  Where to fix: {where}\n  Expected:     {example}\n  To recover:   {remedy}"
    )


def parse_raw_response(raw_response: str, asked_fields: list[str]) -> dict[str, str]:
    """Parse one model response into {field: value} over the asked fields.

    Tries JSON first, then falls back to ``KEY: value`` lines with markdown
    stripped. Fields the model did not emit default to NOT_FOUND. Keys the
    model emitted but nobody asked for are dropped here and reported by the
    reconciliation cell.

    Args:
        raw_response: The model's raw text.
        asked_fields: Field names the prompt requested for this document type.

    Returns:
        One entry per asked field.
    """
    parsed: dict[str, str] = {}
    text = (raw_response or "").strip()

    if text.startswith("{"):
        try:
            loaded = json.loads(text)
            if isinstance(loaded, dict):
                parsed = {str(k): str(v) for k, v in loaded.items()}
        except json.JSONDecodeError:
            parsed = {}

    if not parsed:
        for line in text.split("\n"):
            match = match_key_line(line)
            if match and match.group(1) not in parsed:
                parsed[match.group(1)] = match.group(2).strip()

    return {field: parsed.get(field, NOT_FOUND) or NOT_FOUND for field in asked_fields}


def emitted_keys(raw_response: str) -> set[str]:
    """Every field-like key the model emitted, asked for or not."""
    text = (raw_response or "").strip()
    if text.startswith("{"):
        try:
            loaded = json.loads(text)
            if isinstance(loaded, dict):
                return {str(k) for k in loaded}
        except json.JSONDecodeError:
            pass
    return {m.group(1) for m in (match_key_line(ln) for ln in text.split("\n")) if m}


def read_raw_extractions(path: Path) -> list[dict]:
    """Read raw_extractions.jsonl, skipping records that carry an error."""
    if not path.exists():
        _fail(
            f"No raw extractions file at {path}.",
            "RAW_PATH in the config cell above",
            'RAW_PATH = Path("~/runs/output/raw_extractions.jsonl").expanduser()',
            "point RAW_PATH at the extract stage's output and re-run this cell",
        )

    records, errored = [], []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            record = json.loads(line)
            if record.get("error"):
                errored.append(record.get("image_name", "<unnamed>"))
                continue
            records.append(record)

    if not records:
        _fail(
            f"No usable records in {path} — every record carried an error.",
            str(path),
            "one JSON object per line with image_name, document_type and raw_response",
            "check the extract stage's logs for the failure that produced this file",
        )
    if errored:
        print(f"Skipped {len(errored)} errored extraction(s): {', '.join(errored)}")
    return records


def load_ground_truth(path: Path) -> dict[str, dict]:
    """Load ground truth from a wide CSV or a per-type JSONL, keyed by filename."""
    if not path.exists():
        _fail(
            f"No ground truth file at {path}.",
            "GT_PATH in the config cell above",
            'GT_PATH = Path("~/evaluation_data/ground_truth_extraction.csv").expanduser()',
            "point GT_PATH at the ground truth for this image set and re-run this cell",
        )

    id_columns = ["image_file", "filename", "image_name", "file"]

    if path.suffix == ".jsonl":
        rows = [json.loads(line) for line in path.open(encoding="utf-8") if line.strip()]
    else:
        rows = pd.read_csv(path, dtype=str).fillna(NOT_FOUND).to_dict("records")

    if not rows:
        _fail(
            f"Ground truth file {path} is empty.",
            str(path),
            "at least one row with an image identifier column",
            "regenerate the ground truth file",
        )

    id_column = next((c for c in id_columns if c in rows[0]), None)
    if id_column is None:
        _fail(
            f"No image identifier column in {path}. Found: {sorted(rows[0])[:8]}",
            str(path),
            f"one column named any of {id_columns}",
            f"rename the filename column to '{id_columns[0]}' and re-run",
        )

    return {str(row[id_column]): {k: v for k, v in row.items() if k != id_column} for row in rows}


def match_ground_truth(image_name: str, ground_truth: dict[str, dict]) -> dict | None:
    """Look a document up by filename, falling back to a stem match."""
    if image_name in ground_truth:
        return ground_truth[image_name]
    stem = Path(image_name).stem
    for key, row in ground_truth.items():
        if Path(key).stem == stem:
            return row
    return None

In [ ]:
def split_items(value: str) -> list[str]:
    """Split a list-valued field on LIST_SEPARATOR, dropping blanks."""
    return [item.strip() for item in str(value).split(LIST_SEPARATOR) if item.strip()]


def is_missing(value) -> bool:
    """Whether a value means 'absent' — NOT_FOUND, blank or null."""
    return value is None or str(value).strip().upper() in {NOT_FOUND, "", "NAN", "NONE"}


def parse_date(value: str):
    """Parse a date into a comparable (year, month, day), or None."""
    text = str(value).strip()
    for fmt in DATE_FORMATS:
        try:
            parsed = datetime.strptime(text, fmt)
        except ValueError:
            continue
        return (parsed.year, parsed.month, parsed.day)
    return None


def parse_money(value: str):
    """Parse a monetary or numeric value into a float, or None."""
    cleaned = re.sub(r"[^0-9.\-]", "", str(value).replace(",", ""))
    if cleaned in {"", "-", ".", "-."}:
        return None
    try:
        return float(cleaned)
    except ValueError:
        return None


def values_match(field_type: str, predicted: str, truth: str) -> bool:
    """Compare one predicted value against one ground-truth value."""
    base = field_type.replace("_list", "")

    if base in {"monetary", "numeric"}:
        left, right = parse_money(predicted), parse_money(truth)
        if left is None or right is None:
            return str(predicted).strip() == str(truth).strip()
        return abs(left - right) <= MONETARY_TOLERANCE

    if base in {"date", "date_range"}:
        left, right = parse_date(predicted), parse_date(truth)
        if left and right:
            return left == right
        # Date ranges and unparseable dates fall back to normalised text.
        return re.sub(r"\s+", " ", str(predicted).strip().lower()) == re.sub(
            r"\s+", " ", str(truth).strip().lower()
        )

    if base == "boolean":
        truthy = {"true", "yes", "y", "1"}
        falsy = {"false", "no", "n", "0"}
        left, right = str(predicted).strip().lower(), str(truth).strip().lower()
        if left in truthy | falsy and right in truthy | falsy:
            return (left in truthy) == (right in truthy)
        return left == right

    if base == "id":
        return re.sub(r"\D", "", str(predicted)) == re.sub(r"\D", "", str(truth))

    if base == "exact":
        return str(predicted).strip().upper() == str(truth).strip().upper()

    left = re.sub(r"\s+", " ", str(predicted).strip().lower())
    right = re.sub(r"\s+", " ", str(truth).strip().lower())
    if left == right:
        return True
    return SequenceMatcher(None, left, right).ratio() >= FUZZY_THRESHOLD


def normalise_order(data: dict[str, str], doc_type: str) -> dict[str, str]:
    """Sort parallel transaction lists chronologically.

    Applied to BOTH the extraction and the ground truth before position-wise
    scoring, so that document order versus chronological order is not charged
    as an extraction error. Fields whose item count disagrees with the sort
    key's are left alone.

    Args:
        data: Field values for one document.
        doc_type: Document type deciding which sort group applies.

    Returns:
        A shallow copy with the affected fields re-ordered, or ``data`` when
        no normalisation applies.
    """
    group = TRANSACTION_SORT_GROUPS.get(str(doc_type).upper())
    if not group:
        return data

    sort_value = data.get(group["sort_key"], NOT_FOUND)
    if is_missing(sort_value) or LIST_SEPARATOR not in str(sort_value):
        return data

    dates = [parse_date(item) or (9999, 99, 99) for item in split_items(sort_value)]
    order = sorted(range(len(dates)), key=lambda i: dates[i])
    if order == list(range(len(order))):
        return data

    result = dict(data)
    for field in group["fields"]:
        value = data.get(field, NOT_FOUND)
        if is_missing(value) or LIST_SEPARATOR not in str(value):
            continue
        items = split_items(value)
        if len(items) != len(order):
            continue
        result[field] = f" {LIST_SEPARATOR} ".join(items[i] for i in order)
    return result


def filter_gt_rows(truth: dict[str, str], doc_type: str) -> dict[str, str]:
    """Drop ground-truth rows the prompt never asks for.

    Applies the GT_ROW_FILTERS mask for this document type: positions where the
    ``require_present`` field is NOT_FOUND are removed from every parallel
    field. Those rows are not extraction misses — they were never requested —
    and leaving them in shifts every later position, which position-wise
    scoring charges as a total failure.

    Args:
        truth: Ground-truth values for one document.
        doc_type: Document type deciding which filter applies.

    Returns:
        A shallow copy with the filtered fields shortened, or ``truth`` when no
        filter applies.
    """
    spec = GT_ROW_FILTERS.get(str(doc_type).upper())
    if not spec:
        return truth

    mask_value = truth.get(spec["require_present"], NOT_FOUND)
    if is_missing(mask_value):
        return truth

    mask_items = split_items(mask_value)
    keep = [i for i, item in enumerate(mask_items) if not is_missing(item)]
    if len(keep) == len(mask_items):
        return truth

    result = dict(truth)
    for field in spec["fields"]:
        value = truth.get(field, NOT_FOUND)
        if is_missing(value):
            continue
        items = split_items(value)
        if len(items) != len(mask_items):
            # Parallel fields must line up for the mask to be meaningful.
            continue
        result[field] = f" {LIST_SEPARATOR} ".join(items[i] for i in keep)
    return result


def score_field(field: str, predicted, truth) -> dict:
    """Score one field, returning tp/fp/fn and F1.

    Single-valued fields score 1 or 0. List fields are compared position-wise
    after order normalisation: tp is the count of matching positions, fp the
    surplus predicted items, fn the missing ones.

    A field whose ground truth is NOT_FOUND is still scored — answering
    NOT_FOUND is correct (F1 1.0, no counts, so it carries the macro table but
    contributes nothing to pooled micro counts), and emitting any value is a
    false positive.

    Args:
        field: Field name, used to look up its comparison type.
        predicted: The model's value.
        truth: The ground-truth value.

    Returns:
        Mapping with f1_score, tp, fp and fn.
    """
    field_type = FIELD_TYPES.get(field, DEFAULT_FIELD_TYPE)
    truth_missing, predicted_missing = is_missing(truth), is_missing(predicted)

    if field_type.endswith("_list"):
        truth_items = [] if truth_missing else split_items(truth)
        predicted_items = [] if predicted_missing else split_items(predicted)
        tp = sum(
            1 for left, right in zip(predicted_items, truth_items) if values_match(field_type, left, right)
        )
        fp, fn = len(predicted_items) - tp, len(truth_items) - tp
    else:
        if truth_missing and predicted_missing:
            tp = fp = fn = 0
        elif truth_missing:
            tp, fp, fn = 0, 1, 0
        elif predicted_missing:
            tp, fp, fn = 0, 0, 1
        elif values_match(field_type, predicted, truth):
            tp, fp, fn = 1, 0, 0
        else:
            tp, fp, fn = 0, 1, 1

    denominator = 2 * tp + fp + fn
    # Nothing predicted and nothing expected is a correct answer, not a gap.
    f1 = 1.0 if denominator == 0 else 2 * tp / denominator
    return {"f1_score": f1, "tp": tp, "fp": fp, "fn": fn}


def ground_truth_columns(ground_truth: dict[str, dict]) -> set[str]:
    """Every field name the ground-truth FILE carries, pooled over all its records.

    Pooled over the file rather than read per record, because the two ground-truth
    formats this repo writes disagree per record. The CSV carries the union of every
    type's columns on every row, padded with NOT_FOUND; the JSONL carries only the
    fields that record's own document type is asked for. Deciding the scored set from
    one record's keys therefore makes the JSONL score a MISCLASSIFIED document on
    fewer fields than the CSV does — it silently drops the fields the wrong prompt
    asked for, which are exactly the ones that catch the hallucination.

    Args:
        ground_truth: Mapping from load_ground_truth.

    Returns:
        The union of field names over every record in the file.
    """
    return {key for row in ground_truth.values() for key in row}


def scored_fields_for(predicted_type: str, gt_columns: set[str]) -> list[str]:
    """Fields to score: the prompt's asked set, minus legacy, minus any the file lacks.

    The two exclusions are different things, and only one of them is a gap:

    * a field the ground-truth FILE never carries cannot be scored at all — that is a
      schema gap, and the reconciliation cell above reports it;
    * a field the file carries but this record omits is NOT a gap. It is NOT_FOUND,
      a legitimate expected answer, and build_records fills it as one.

    Args:
        predicted_type: The document type the model predicted, which decides which
            questions it was asked.
        gt_columns: Field names from ground_truth_columns.

    Returns:
        The field names to score for this document, in prompt order.
    """
    asked = FIELDS_ASKED_BY_DOC_TYPE.get(str(predicted_type).upper())
    if asked is None:
        _fail(
            f"No field list declared for predicted document type '{predicted_type}'.",
            "FIELDS_ASKED_BY_DOC_TYPE in the config cell above",
            f'FIELDS_ASKED_BY_DOC_TYPE["{str(predicted_type).upper()}"] = ["DOCUMENT_TYPE", ...]',
            "transcribe that type's field list from your prompt file and re-run this cell",
        )
    return [f for f in asked if f not in LEGACY_FIELDS and f in gt_columns]

In [ ]:
def build_records(
    raw_records: list[dict], ground_truth: dict[str, dict], *, source: Path
) -> tuple[list[dict], list[str]]:
    """Parse, score and assemble one result record per document.

    Emits the same shape the pipeline's evaluation_results.jsonl uses, so every
    analysis cell below is unchanged, plus predicted_document_type for the
    confusion table. ``document_type`` is the GROUND TRUTH type, so a
    misclassified document is grouped under what it really is.

    Args:
        raw_records: Records from read_raw_extractions.
        ground_truth: Mapping from load_ground_truth.
        source: The file raw_records came from, named in the failure diagnostic.
            Passed explicitly rather than read from RAW_PATH so the comparison
            section's second run cites its own file and not the first one's.

    Returns:
        The scored records, and the names of images with no ground truth.
    """
    results, unmatched = [], []

    # Once per file, not once per record — see ground_truth_columns for why the
    # distinction changes the score of a misclassified document.
    gt_columns = ground_truth_columns(ground_truth)

    for record in raw_records:
        image_name = record["image_name"]
        gt_row = match_ground_truth(image_name, ground_truth)
        if gt_row is None:
            unmatched.append(image_name)
            continue

        predicted_type = str(record["document_type"]).upper()
        fields = scored_fields_for(predicted_type, gt_columns)

        extracted = parse_raw_response(record.get("raw_response", ""), fields)
        gt_type = str(gt_row.get("DOCUMENT_TYPE", predicted_type)).upper()

        # Drop GT rows the prompt never asked for, then sort both sides. Both
        # keyed on the GT type: it decides what the document really is.
        truth = filter_gt_rows({f: gt_row.get(f, NOT_FOUND) for f in fields}, gt_type)
        extracted = normalise_order(extracted, gt_type)
        truth = normalise_order(truth, gt_type)

        field_scores = {f: score_field(f, extracted[f], truth[f]) for f in fields}
        f1_values = pd.Series([s["f1_score"] for s in field_scores.values()])

        results.append(
            {
                "image_name": image_name,
                "document_type": gt_type,
                "predicted_document_type": predicted_type,
                "field_scores": field_scores,
                "overall_accuracy": f1_values.mean(),
                "median_f1": f1_values.median(),
            }
        )

    if not results:
        _fail(
            "No document matched the ground truth by filename.",
            f"{source} and {GT_PATH}",
            "image_name values in the raw file matching the GT identifier column",
            "check that both files describe the same image set",
        )
    return results, unmatched


def field_frame(records: list[dict]) -> pd.DataFrame:
    """Flatten records into one row per (image, field).

    Args:
        records: Scored records from build_records.

    Returns:
        Long-format frame with image_name, document_type, field, f1_score, tp, fp, fn.
    """
    return pd.DataFrame(
        [
            {
                "image_name": record["image_name"],
                "document_type": record["document_type"],
                "predicted_document_type": record["predicted_document_type"],
                "field": field,
                "f1_score": field_score["f1_score"],
                "tp": field_score["tp"],
                "fp": field_score["fp"],
                "fn": field_score["fn"],
            }
            for record in records
            for field, field_score in record["field_scores"].items()
        ]
    )


def document_frame(records: list[dict]) -> pd.DataFrame:
    """Build one row per document with the two per-document F1 aggregates.

    Args:
        records: Scored records from build_records.

    Returns:
        Frame with image_name, document_type, predicted_document_type, n_fields,
        doc_mean_f1 and doc_median_f1.
    """
    return pd.DataFrame(
        [
            {
                "image_name": record["image_name"],
                "document_type": record["document_type"],
                "predicted_document_type": record["predicted_document_type"],
                "n_fields": len(record["field_scores"]),
                "doc_mean_f1": record["overall_accuracy"],
                "doc_median_f1": record["median_f1"],
            }
            for record in records
        ]
    )


raw_records = read_raw_extractions(RAW_PATH)
ground_truth = load_ground_truth(GT_PATH)
records, unmatched = build_records(raw_records, ground_truth, source=RAW_PATH)

if unmatched:
    print(f"No ground truth for {len(unmatched)} image(s): {', '.join(unmatched[:5])}")

scores = field_frame(records)
documents = document_frame(records)

print(f"{len(documents)} documents scored, {scores['field'].nunique()} distinct fields")
print(documents["document_type"].value_counts().to_string())
documents.head()

## Before the scores: what was classified as what, and what got scored

The pipeline's `evaluation_results.jsonl` records only the **predicted** document type, so a
misclassified document is filed under the wrong type and its true type is under-reported. Reading
from raw output plus ground truth makes both visible, so these two checks come first.

**Classification.** Every table below groups on the **ground truth** type. Where the two disagree,
the extraction was driven by the *predicted* type's prompt — so the field list follows the
prediction, while the answer comes from the truth.

**Field reconciliation.** Three sets can disagree, and only one of them decides scoring:

* **asked** — from `FIELDS_ASKED_BY_DOC_TYPE`, i.e. the prompt. **This is what gets scored.**
* **emitted** — keys the model actually produced. A key here but not in *asked* was invented by
  the model; it is not evidence of anything except that the prompt is not being followed.
* **in GT** — columns the ground truth carries. A field here but not in *asked* has real ground
  truth the prompt never requests, so the pipeline cannot reach it. Scoring it would charge the
  model for a schema gap rather than an extraction failure.

In [ ]:
# Predicted vs actual document type. Off-diagonal entries are misclassifications;
# they cost the DOCUMENT_TYPE field, and cost the rest only where the two types'
# prompts ask for different fields.
confusion = (
    documents.groupby(["document_type", "predicted_document_type"])
    .size()
    .rename("n")
    .reset_index()
    .pivot(index="document_type", columns="predicted_document_type", values="n")
    .fillna(0)
    .astype(int)
)

misclassified = int((documents["document_type"] != documents["predicted_document_type"]).sum())
print(f"{misclassified}/{len(documents)} documents misclassified")
display(confusion)


def reconcile_fields(raw_records: list[dict], ground_truth: dict[str, dict]) -> pd.DataFrame:
    """Compare the asked, emitted and ground-truth field sets per document type.

    Args:
        raw_records: Records from read_raw_extractions.
        ground_truth: Mapping from load_ground_truth.

    Returns:
        One row per (document type, field) that is not scored, saying why.
    """
    rows = []
    for gt_type in sorted({str(r.get("DOCUMENT_TYPE", "")).upper() for r in ground_truth.values()}):
        members = [
            r
            for r in raw_records
            if (match_ground_truth(r["image_name"], ground_truth) or {}).get("DOCUMENT_TYPE", "").upper()
            == gt_type
        ]
        if not members:
            continue

        gt_rows = [match_ground_truth(r["image_name"], ground_truth) for r in members]
        asked = set()
        for record in members:
            asked |= set(FIELDS_ASKED_BY_DOC_TYPE.get(str(record["document_type"]).upper(), []))
        emitted = set().union(*(emitted_keys(r.get("raw_response", "")) for r in members))
        in_gt = {k for row in gt_rows for k in row}
        populated = {k for row in gt_rows for k in row if not is_missing(row[k])}

        for field in sorted((emitted | in_gt) - (asked - LEGACY_FIELDS)):
            if field in LEGACY_FIELDS:
                reason = "legacy field — never scored"
            elif field in emitted and field not in in_gt:
                reason = "emitted but not asked, and no GT column"
            elif field in populated and field not in asked:
                reason = "real GT, never asked — pipeline cannot reach it"
            elif field not in asked:
                reason = "not asked for this type"
            else:
                continue
            n_emitted = sum(1 for r in members if field in emitted_keys(r.get("raw_response", "")))
            n_populated = sum(1 for row in gt_rows if not is_missing(row.get(field)))
            rows.append(
                {
                    "document_type": gt_type,
                    "field": field,
                    "asked": field in asked,
                    "emitted": f"{n_emitted}/{len(members)}",
                    "gt_populated": f"{n_populated}/{len(gt_rows)}",
                    "why_not_scored": reason,
                }
            )
    return pd.DataFrame(rows)


unscored = reconcile_fields(raw_records, ground_truth)
if unscored.empty:
    print("\nEvery emitted and ground-truth field is in the asked set.")
else:
    print("\nFields present in the data but NOT scored:")
    display(unscored)

In [ ]:
# The reconciliation above reads "real GT, never asked" as a property of a whole
# document type. That only holds if a field is populated for every document of a
# type or for none. Verify it rather than assuming — a field populated for, say,
# 30 of 55 documents is genuinely optional, and calling it a schema gap would be
# wrong.
def check_population_bimodal(ground_truth: dict[str, dict]) -> pd.DataFrame:
    """Report any (type, field) whose GT population rate is neither 0% nor 100%.

    Args:
        ground_truth: Mapping from load_ground_truth.

    Returns:
        One row per partially-populated (document type, field).
    """
    frame = pd.DataFrame(list(ground_truth.values()))
    if "DOCUMENT_TYPE" not in frame:
        return pd.DataFrame()

    rows = []
    for gt_type, group in frame.groupby(frame["DOCUMENT_TYPE"].str.upper()):
        for field in group.columns:
            rate = group[field].map(lambda v: not is_missing(v)).mean()
            if 0.0 < rate < 1.0:
                rows.append(
                    {
                        "document_type": gt_type,
                        "field": field,
                        "populated": f"{int(rate * len(group))}/{len(group)}",
                        "rate": round(rate, 3),
                    }
                )
    return pd.DataFrame(rows)


partial = check_population_bimodal(ground_truth)
if partial.empty:
    print("Ground truth is bimodal: every field is populated for all or none of each type.")
else:
    display(partial)
    _fail(
        f"{len(partial)} (document type, field) pair(s) are only partially populated in the "
        "ground truth, so 'this field belongs to this type' has no answer for them.",
        f"{GT_PATH} — the fields listed in the table above",
        "each field populated for every document of a type, or NOT_FOUND for all of them",
        "fix the ground truth, or declare the field explicitly in FIELDS_ASKED_BY_DOC_TYPE and re-run",
    )

## The two headline numbers, plus their spread

Both rows below are **means across documents**. The `(mean)` / `(median)` in the label refers to
how the *fields within one document* were collapsed, not to how documents were combined:

* `Avg F1 (mean)` — mean over documents of each document's **mean** field F1
* `Avg F1 (median)` — mean over documents of each document's **median** field F1

So `Avg F1 (median)` is not the median of anything at the corpus level. It reads higher than the
mean whenever a minority of fields fail badly, because the within-document median discards them —
that gap is the point of reporting both.

The `std` column is the sample standard deviation (ddof=1) of the per-document scores. Quote it:
a centre without a spread is not a result.

These are **this notebook's** figures, not the pipeline's — see the caveat at the top.

In [ ]:
DOC_METRICS = {"Avg F1 (mean)": "doc_mean_f1", "Avg F1 (median)": "doc_median_f1"}


def execution_summary(docs: pd.DataFrame) -> pd.DataFrame:
    """Reproduce the pipeline's headline F1 rows and add their dispersion.

    Args:
        docs: Per-document frame from document_frame.

    Returns:
        One row per headline metric with n_images, mean, std, median, min and max
        taken across documents. The mean column is the value the pipeline prints.
    """
    summary = pd.DataFrame(
        {
            "n_images": docs[column].count(),
            "mean": docs[column].mean(),
            "std": docs[column].std(),
            "median": docs[column].median(),
            "min": docs[column].min(),
            "max": docs[column].max(),
        }
        for column in DOC_METRICS.values()
    )
    summary.index = pd.Index(DOC_METRICS.keys(), name="Metric")
    return summary


summary = execution_summary(documents)

# Printed the way the pipeline prints it, to 3 dp, so the two can be compared row by row.
for metric, row in summary.iterrows():
    print(f"{metric:<18} {row['mean']:.3f}   (sd {row['std']:.3f}, n={int(row['n_images'])})")

summary.round(4)

In [ ]:
# The same two headline metrics split by document type. The pipeline prints only the
# corpus-wide figure, so a run whose doc-type mix changes moves the headline for reasons
# that have nothing to do with model quality — this table shows which type is responsible.
summary_by_type = (
    documents.groupby("document_type")[list(DOC_METRICS.values())].agg(["count", "mean", "std"]).round(4)
)
summary_by_type

## Three traps in the headline figures

These are properties of how the two numbers are constructed. They all push in the **optimistic**
direction, so a headline read at face value overstates the model. The cell below measures traps 1
and 2 on whatever file you loaded, rather than assuming values from any particular run.

### 1. The scale does not start at zero

A document's mean F1 gives **each field equal weight regardless of how much work it represents**.
A single date range and a 31-row transaction table are worth the same 1/n.

That matters because some fields are effectively free — any field that scores 1.0 on every
document of its type adds a constant to the score before the hard fields are read. Two kinds
qualify, and both are measured below rather than assumed:

* fields the model gets right every time;
* fields whose ground truth is `NOT_FOUND` for a whole document type, where answering
  `NOT_FOUND` is correct. These are *not* free in principle — emitting a value is a false
  positive, which is exactly how a supplier name landing in `PAYER_NAME` gets caught — but they
  are free in practice whenever the model never hallucinates on them.

`DOCUMENT_TYPE` is **not** in this category by construction. It is scored like any other field,
the model's extracted value against the ground truth, so it fails whenever the document is
misclassified. Check the confusion table above before assuming it is free: where classification
is imperfect, this field is doing real work.

To rescale onto the range the model can actually influence, use `(score - floor) / (1 - floor)`,
taking the floor from `floor_on_doc_mean_f1` below. It depends on the field schema and on which
fields happen to be saturated in *your* run, and for a mixed-document run it is a
document-weighted blend of the per-type floors — so do not carry a floor between runs.

### 2. `Avg F1 (median)` can be the *best* of the working fields, not the middle of anything

With five fields of which the top two are always 1.0, the sorted scores look like
`[a, b, c, 1.0, 1.0]` and the median is `c` — the **maximum** of the three fields doing real work.
Where that holds, `Avg F1 (median)` is a best-case-per-document statistic, and any gap over
`Avg F1 (mean)` is not robustness but the free fields propping up the middle.

How often it holds depends on the field count and on which fields are saturated, so it does not
transfer between document types. `median_equals_best_worked` below measures it per type.

### 3. The "median" row can be the *less* stable of the two

Compare the two `std` values in the summary above. A max-of-k over a near-bimodal distribution is
a step function that snaps between 1.0 and near-zero, so it can move *more* between documents than
an average does — inverting the usual intuition that medians are steadier.

What is structural is only this: `Avg F1 (median)` discards the weakest fields by construction, so
reaching for it as the "safer, more conservative" number is unjustified in either direction. If
you want robustness across documents, use the `median` column of the summary table — that one is a
genuine corpus-level median.

In [ ]:
def scale_diagnostics(frame: pd.DataFrame, docs: pd.DataFrame) -> pd.DataFrame:
    """Quantify traps 1 and 2 for whatever file is loaded, per document type.

    Trap 1: a field scoring 1.0 on every document of its type adds a constant to every
    doc_mean_f1, putting a floor on the metric. The floor is that field count over the
    total field count.

    Trap 2: whether median_f1 collapses to "best of the fields that do real work" is
    decided empirically per document rather than by index arithmetic, so it stays correct
    for any field count or schema.

    Args:
        frame: Long-format frame from field_frame.
        docs: Per-document frame from document_frame.

    Returns:
        One row per document type: field counts, the implied floor, the worst observed
        score, how often the median equals the best worked field, and the free fields.
    """
    rows, index = [], []
    for doc_type, group in frame.groupby("document_type"):
        per_field_min = group.groupby("field")["f1_score"].min()
        free = set(per_field_min[per_field_min == 1.0].index)
        worked = group[~group["field"].isin(free)]

        best_worked = worked.groupby("image_name")["f1_score"].max()
        medians = docs.set_index("image_name")["doc_median_f1"].reindex(best_worked.index)
        equals_best = (medians - best_worked).abs() < 1e-9

        index.append(doc_type)
        rows.append(
            {
                "n_fields": len(per_field_min),
                "n_always_perfect": len(free),
                "floor_on_doc_mean_f1": len(free) / len(per_field_min),
                "worst_doc_mean_f1": docs.loc[docs["document_type"] == doc_type, "doc_mean_f1"].min(),
                "median_equals_best_worked": equals_best.mean(),
                "always_perfect_fields": ", ".join(sorted(free)) or "(none)",
            }
        )
    return pd.DataFrame(rows, index=pd.Index(index, name="document_type"))


diagnostics = scale_diagnostics(scores, documents)

for doc_type, row in diagnostics.iterrows():
    print(
        f"{doc_type}: {row.n_always_perfect}/{row.n_fields} fields perfect on every document"
        f" -> doc_mean_f1 floor {row.floor_on_doc_mean_f1:.3f}"
        f" (worst observed {row.worst_doc_mean_f1:.3f})"
    )
    if row.median_equals_best_worked > 0:
        print(
            f"    trap 2: median_f1 equals the BEST worked field on "
            f"{row.median_equals_best_worked:.0%} of these documents"
        )

diagnostics.round(4)

In [ ]:
MACRO_COLS = ["n_docs", "mean_f1", "std_f1", "min_f1", "max_f1"]
MICRO_COLS = ["n_items", "tp", "fp", "fn", "micro_precision", "micro_recall", "micro_f1"]


def summarise(frame: pd.DataFrame, *, by: list[str]) -> pd.DataFrame:
    """Aggregate macro and micro F1 statistics over the given grouping columns.

    Args:
        frame: Long-format frame from field_frame.
        by: Columns to group on, e.g. ["document_type", "field"].

    Returns:
        One row per group holding both the MACRO_COLS and MICRO_COLS statistics.
        std_f1 is the sample standard deviation (ddof=1) of the per-document F1
        scores, so it is NaN for any group holding a single document.
    """
    summary = frame.groupby(by, dropna=False).agg(
        n_docs=("image_name", "nunique"),
        mean_f1=("f1_score", "mean"),
        std_f1=("f1_score", "std"),
        min_f1=("f1_score", "min"),
        max_f1=("f1_score", "max"),
        tp=("tp", "sum"),
        fp=("fp", "sum"),
        fn=("fn", "sum"),
    )
    summary["n_items"] = summary["tp"] + summary["fn"]
    predicted = summary["tp"] + summary["fp"]
    actual = summary["tp"] + summary["fn"]
    summary["micro_precision"] = (summary["tp"] / predicted).where(predicted > 0, 0.0)
    summary["micro_recall"] = (summary["tp"] / actual).where(actual > 0, 0.0)
    denominator = 2 * summary["tp"] + summary["fp"] + summary["fn"]
    summary["micro_f1"] = (2 * summary["tp"] / denominator).where(denominator > 0, 0.0)
    return summary[MACRO_COLS + MICRO_COLS]


overall = summarise(scores, by=["field"]).sort_values("mean_f1")
per_type = summarise(scores, by=["document_type", "field"]).sort_values(["document_type", "mean_f1"])
per_type.shape, overall.shape

## Macro table — every document counts once

`mean_f1` is the average of the per-document F1 scores. This is the number to quote when you
want "how well does the model do on a typical document". It is insensitive to table length,
so a one-line receipt and a 31-transaction bank statement carry equal weight.

In [ ]:
macro_table = overall[MACRO_COLS].round(4)
macro_table.insert(
    1,
    "mean_pm_std",
    overall.apply(lambda r: f"{r.mean_f1:.3f} ± {r.std_f1:.3f}", axis=1),
)

macro_by_type = per_type[MACRO_COLS].round(4)

display(macro_table)
display(macro_by_type)

## Micro table — every extracted item counts once

`micro_f1` pools the raw `tp`/`fp`/`fn` counts across documents before computing the score, so
a 31-transaction statement weighs 31x a single-transaction one. This is the number to quote for
"what fraction of all transactions in the corpus did we get right". `n_items` is the ground-truth
item count (`tp + fn`) behind each row — check it before trusting a field's micro score.

Precision and recall are broken out because they fail differently: low recall means truncated
extraction, low precision means hallucinated rows.

In [ ]:
micro_table = overall[MICRO_COLS].sort_values("micro_f1").round(4)
micro_by_type = per_type[MICRO_COLS].sort_values(["document_type", "micro_f1"]).round(4)

display(micro_table)
display(micro_by_type)

In [ ]:
# Mean F1 (and its spread) as a field x document-type matrix.
# NaN = field not evaluated for that document type.
matrix = per_type["mean_f1"].unstack("document_type")
errors = per_type["std_f1"].unstack("document_type")
matrix.round(4)

In [ ]:
# Categorical hues assigned per DOCUMENT TYPE rather than per column position, so
# scoring a corpus that happens to lack a type does not repaint the survivors.
# Validated for colour-vision deficiency; the legend carries identity, so colour is
# never the only channel a reader has.
DOC_TYPE_COLOURS = {
    "BANK_STATEMENT": "#1D4ED8",
    "INVOICE": "#C2410C",
    "RECEIPT": "#047857",
}

undeclared = [doc_type for doc_type in matrix.columns if doc_type not in DOC_TYPE_COLOURS]
if undeclared:
    _fail(
        f"No colour declared for document type(s): {', '.join(undeclared)}.",
        "DOC_TYPE_COLOURS in this cell",
        f'DOC_TYPE_COLOURS["{undeclared[0]}"] = "#7C3AED"',
        "add one entry per document type rather than letting matplotlib cycle its defaults",
    )

# Error bars are +/- 1 standard deviation across documents, so they clip outside [0, 1].
# The white edge is a separator, not decoration: without it the three bars in a group
# abut and a long field reads as one wide bar of an indeterminate colour.
ax = matrix.plot.barh(
    figsize=(9, 0.45 * len(matrix) + 2),
    xlim=(0, 1),
    width=0.75,
    color=[DOC_TYPE_COLOURS[doc_type] for doc_type in matrix.columns],
    edgecolor="white",
    linewidth=0.8,
    xerr=errors.fillna(0.0),
    capsize=3,
    error_kw={"elinewidth": 1, "ecolor": "0.45"},
)
ax.set_xlabel("mean F1 (± 1 sd)")
ax.set_ylabel("")
ax.set_title(f"Mean F1 per field — {RAW_PATH.name}", fontsize=12)
ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)
for spine in ("top", "right", "left"):
    ax.spines[spine].set_visible(False)

# Below the axes rather than inside it: every bar reaches 1.0 on a clean run, so an
# inset legend sits on top of the data it is labelling.
ax.legend(
    title="document type",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.06),
    ncol=len(matrix.columns),
    frameon=False,
)
ax.figure.tight_layout()

## Where the loss actually is

The micro table above separates precision from recall for a reason. In the 15-document run this
notebook was developed against, `micro_precision` is **exactly 1.0 on every field** while recall
runs 0.59–0.71. Check whether that still holds on your file, because it decides what to fix:

* **precision 1.0, recall low** — the model is *truncating*. It stops emitting transactions
  partway down the table and never invents one. The fix is in prompting, tiling or context
  length, not in output validation.
* **precision low** — the model is *hallucinating* rows, or the cleaner is mangling them. A very
  different investigation.

Collapsing these into a single `micro_f1` hides the distinction completely, which is why the
micro table is a separate table rather than one more column on the macro one.

The table below lists the worst (document, field) pairs. Expect the loss to be concentrated
rather than spread: in the reference run a single document contributed 31 of the missing items,
and removing it moves the headline more than any per-field prompt change would.

In [ ]:
# Worst documents per field — the usual starting point for error analysis.
scores.sort_values("f1_score").head(15)[
    ["document_type", "field", "image_name", "f1_score", "tp", "fp", "fn"]
]

## Clean vs degraded — the same documents, twice

`python -m generators.pipeline eval-set` writes the clean and the degraded images into two
directories with **identical filenames and byte-identical ground truth**, so every document
appears once in each and differs only in image quality. Point `RAW_PATH_B` at the degraded run's
`raw_extractions.jsonl` and the cells below difference the two. With `RAW_PATH_B = None` they
print a line saying so and do nothing else.

Only receipts are degraded — bank statements and invoices arrive as clean PDFs or printouts, so
degrading them models nothing. Expect the other two types to show a delta of zero, and treat a
non-zero one as evidence of a non-deterministic model rather than of damage.

**This is a paired comparison, and that is the whole point.** Do not read the two runs' headline
F1 side by side: that difference mixes the degradation effect with whatever the document mix
contributes. Differencing *per document* cancels the document out — a receipt that is hard when
clean is equally hard when degraded — so what survives the subtraction is the damage.

Three things to read, in order:

* **Coverage.** A difference is defined only on documents scored in *both* runs. Anything scored
  in one is reported and then excluded; silently dropping it would let a run that died halfway
  through look like a run that got easier.
* **Classification shift.** Degradation can change the *predicted* document type, and that changes
  which fields the model was asked for. Where it happens the two runs are no longer answering the
  same questions, so those (document, field) pairs leave the paired table and are counted
  separately rather than being differenced against nothing.
* **Per-field delta.** Negative means the degraded run scored worse. This is where the number you
  actually want lives: not "degradation cost 4 points" but "degradation cost 4 points, all of it
  in `LINE_ITEM_DESCRIPTIONS`".

`mean_delta` is a mean of per-document differences. It is on the F1 scale but is **not an F1** —
it is a difference of two, it can be negative, and it is not comparable to the macro or micro
tables above. Quote it with its `sd_delta` and its `n_docs`, and prefer the per-type rows: an
ALL-row delta over a corpus where only receipts were degraded is diluted by two thirds.

In [ ]:
def score_run(path: Path, gt: dict[str, dict]) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """Parse and score one raw-extractions file against an already-loaded ground truth.

    Exactly what the build cell does for RAW_PATH, factored out so a second run is
    scored identically — same ground truth, same field sets, same matching knobs.
    That sameness is what lets the difference be attributed to the images.

    Args:
        path: A raw_extractions.jsonl to score.
        gt: Mapping from load_ground_truth.

    Returns:
        Its long-format field frame, its per-document frame, and the names of
        images in the file that had no ground truth.
    """
    run_records, run_unmatched = build_records(read_raw_extractions(path), gt, source=path)
    return field_frame(run_records), document_frame(run_records), run_unmatched


if RAW_PATH_B is None:
    comparison = None
    print("RAW_PATH_B is None — comparison disabled. Set it in the config cell to enable.")
else:
    scores_b, documents_b, unmatched_b = score_run(RAW_PATH_B, ground_truth)
    if unmatched_b:
        print(f"No ground truth for {len(unmatched_b)} image(s) in {RUN_LABELS[1]}: {unmatched_b[:5]}")

    paired_names = sorted(set(documents["image_name"]) & set(documents_b["image_name"]))
    if not paired_names:
        _fail(
            "The two runs share no image name, so no paired difference is defined.",
            f"RAW_PATH ({RAW_PATH.name}) and RAW_PATH_B ({RAW_PATH_B.name}) in the config cell",
            "two runs over the same image set — eval-set writes identical filenames into "
            "its clean and degraded directories precisely so this join works",
            "check RAW_PATH_B is the other half of the same export, not a different corpus",
        )

    print(
        f"{len(paired_names)} documents scored in both runs "
        f"({RUN_LABELS[0]} {len(documents)}, {RUN_LABELS[1]} {len(documents_b)}; "
        f"{len(documents) - len(paired_names)} + {len(documents_b) - len(paired_names)} unpaired, excluded)"
    )

    # Degradation can change the PREDICTED type, which changes which fields were
    # asked. Those documents are still paired, but some of their fields will not be.
    predictions = (
        documents.set_index("image_name")
        .loc[paired_names, ["document_type", "predicted_document_type"]]
        .rename(columns={"predicted_document_type": RUN_LABELS[0]})
        .join(documents_b.set_index("image_name")["predicted_document_type"].rename(RUN_LABELS[1]))
    )
    shifted = predictions[predictions[RUN_LABELS[0]] != predictions[RUN_LABELS[1]]]
    print(f"{len(shifted)}/{len(paired_names)} documents changed predicted type between runs")
    if not shifted.empty:
        display(
            shifted.groupby(["document_type", RUN_LABELS[0], RUN_LABELS[1]])
            .size()
            .rename("n")
            .reset_index()
        )

    comparison = {"scores_b": scores_b, "documents_b": documents_b, "names": paired_names}

In [ ]:
def paired_headline(a: pd.Series, b: pd.Series, doc_types: pd.Series) -> pd.DataFrame:
    """Summarise the per-document difference b - a, per document type and overall.

    Args:
        a: doc_mean_f1 for run A, indexed by image_name.
        b: doc_mean_f1 for run B, indexed by the same names.
        doc_types: Ground-truth document type, indexed by the same names.

    Returns:
        One row per document type plus an ALL row, holding both runs' means, the
        mean paired difference and its spread, and how many documents moved each
        way. n_worse + n_better rarely sums to n_docs — the remainder scored
        identically in both runs, which is itself worth seeing.
    """
    frame = pd.DataFrame({"a": a, "b": b, "delta": b - a, "document_type": doc_types})

    def _agg(group: pd.DataFrame) -> pd.Series:
        return pd.Series(
            {
                "n_docs": len(group),
                RUN_LABELS[0]: group["a"].mean(),
                RUN_LABELS[1]: group["b"].mean(),
                "mean_delta": group["delta"].mean(),
                "sd_delta": group["delta"].std(),
                "n_worse": int((group["delta"] < -1e-9).sum()),
                "n_better": int((group["delta"] > 1e-9).sum()),
            }
        )

    per_type = frame.groupby("document_type")[["a", "b", "delta"]].apply(_agg)
    overall = _agg(frame).to_frame("ALL").T
    overall.index.name = per_type.index.name
    return pd.concat([per_type, overall])


if comparison is None:
    print("Comparison disabled — set RAW_PATH_B in the config cell.")
else:
    a_docs = documents.set_index("image_name").loc[comparison["names"]]
    b_docs = comparison["documents_b"].set_index("image_name").loc[comparison["names"]]
    headline_delta = paired_headline(a_docs["doc_mean_f1"], b_docs["doc_mean_f1"], a_docs["document_type"])

    for doc_type, row in headline_delta.iterrows():
        print(
            f"{doc_type:<15} {row[RUN_LABELS[0]]:.3f} -> {row[RUN_LABELS[1]]:.3f}"
            f"   delta {row['mean_delta']:+.3f} (sd {row['sd_delta']:.3f},"
            f" {int(row['n_worse'])} worse / {int(row['n_better'])} better of {int(row['n_docs'])})"
        )

    headline_delta.round(4)

In [ ]:
import matplotlib.pyplot as plt

# Diverging pair either side of a neutral zero line, validated for colour-vision
# deficiency. The sign is carried by bar direction as well as hue, so the chart
# still reads in greyscale or print.
WORSE, BETTER = "#C2410C", "#1D4ED8"

# A field whose mean delta is under this is treated as not having moved.
MOVED_THRESHOLD = 0.005


def paired_field_delta(
    a: pd.DataFrame, b: pd.DataFrame, names: list[str]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Difference each (document, field) score between the two runs.

    Joined on (image_name, document_type, field). A pair present in only one run is
    NOT differenced: it means the two runs were asked different questions about that
    document, which the classification-shift table above explains. Those rows are
    returned separately rather than dropped, so they cannot quietly shrink a delta.

    Args:
        a: Long-format field frame for run A.
        b: Long-format field frame for run B.
        names: Image names scored in both runs.

    Returns:
        The per-(document type, field) delta table, and the unpaired rows.
    """
    keys = ["image_name", "document_type", "field"]
    columns = keys + ["f1_score", "tp", "fp", "fn"]
    merged = a.loc[a["image_name"].isin(names), columns].merge(
        b.loc[b["image_name"].isin(names), columns],
        on=keys,
        how="outer",
        suffixes=("_a", "_b"),
        indicator=True,
    )

    unpaired = merged[merged["_merge"] != "both"]
    both = merged[merged["_merge"] == "both"].assign(
        delta=lambda frame: frame["f1_score_b"] - frame["f1_score_a"]
    )

    summary = both.groupby(["document_type", "field"]).agg(
        n_docs=("image_name", "nunique"),
        mean_a=("f1_score_a", "mean"),
        mean_b=("f1_score_b", "mean"),
        mean_delta=("delta", "mean"),
        sd_delta=("delta", "std"),
        tp_a=("tp_a", "sum"),
        fp_a=("fp_a", "sum"),
        fn_a=("fn_a", "sum"),
        tp_b=("tp_b", "sum"),
        fp_b=("fp_b", "sum"),
        fn_b=("fn_b", "sum"),
    )
    for side in ("a", "b"):
        denominator = 2 * summary[f"tp_{side}"] + summary[f"fp_{side}"] + summary[f"fn_{side}"]
        summary[f"micro_{side}"] = (2 * summary[f"tp_{side}"] / denominator).where(denominator > 0, 0.0)
    summary["micro_delta"] = summary["micro_b"] - summary["micro_a"]

    return (
        summary.rename(
            columns={
                "mean_a": f"mean_{RUN_LABELS[0]}",
                "mean_b": f"mean_{RUN_LABELS[1]}",
                "micro_a": f"micro_{RUN_LABELS[0]}",
                "micro_b": f"micro_{RUN_LABELS[1]}",
            }
        ),
        unpaired,
    )


def plot_field_delta(field_delta: pd.DataFrame, n_paired: int) -> None:
    """Draw the fields that moved, one bar each, sorted worst-first.

    Only fields whose mean delta clears MOVED_THRESHOLD get a bar: a chart of thirty
    zero-length bars buries the two that matter. The count that did not move is
    stated in the subtitle rather than dropped silently — "the other 31 were
    identical in both runs" is part of the finding, not an omission.

    Args:
        field_delta: The per-(document type, field) table from paired_field_delta.
        n_paired: How many documents the deltas were computed over, for the title.
    """
    moved = field_delta[field_delta["mean_delta"].abs() >= MOVED_THRESHOLD]
    if moved.empty:
        print(
            f"No field moved by {MOVED_THRESHOLD} or more across "
            f"{len(field_delta)} scored (document type, field) pairs — nothing to plot."
        )
        return

    # Descending, because barh fills upward: this puts the worst field at the top.
    order = moved.sort_values("mean_delta", ascending=False)
    labels = [f"{doc_type.split('_')[0].title()} · {field}" for doc_type, field in order.index]
    deltas = order["mean_delta"].to_numpy()

    _, ax = plt.subplots(figsize=(9, 0.45 * len(order) + 2.4))
    ax.barh(labels, deltas, height=0.6, color=[WORSE if v < 0 else BETTER for v in deltas])
    ax.axvline(0, color="0.45", linewidth=1)

    # Headroom on both sides so an end label never overruns the axis into the
    # y tick text, which is what happens when the longest bar defines the limit.
    ax.set_xlim(min(deltas.min() * 1.35, -0.02), max(deltas.max() * 1.35, 0.02))

    ax.set_xlabel(f"mean paired delta in F1  ({RUN_LABELS[1]} − {RUN_LABELS[0]})")
    ax.set_title(f"Per-field cost of degradation — {n_paired} paired documents", fontsize=12, pad=26)
    ax.text(
        0.0,
        1.02,
        f"{len(order)} of {len(field_delta)} scored fields moved; "
        f"{len(field_delta) - len(order)} identical in both runs",
        transform=ax.transAxes,
        fontsize=9,
        color="0.35",
        va="bottom",
    )
    ax.grid(axis="x", alpha=0.25)
    ax.set_axisbelow(True)
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)

    for y, value in enumerate(deltas):
        ax.text(
            value,
            y,
            f" {value:+.3f} ",
            va="center",
            ha="right" if value < 0 else "left",
            fontsize=8,
            color="0.25",
        )
    plt.tight_layout()


if comparison is None:
    print("Comparison disabled — set RAW_PATH_B in the config cell.")
else:
    field_delta, unpaired = paired_field_delta(scores, comparison["scores_b"], comparison["names"])

    if not unpaired.empty:
        print(
            f"{len(unpaired)} (document, field) pair(s) scored in only one run — excluded from the "
            "delta. These are the classification shifts above changing which fields were asked."
        )

    display(
        field_delta[
            [
                "n_docs",
                f"mean_{RUN_LABELS[0]}",
                f"mean_{RUN_LABELS[1]}",
                "mean_delta",
                "sd_delta",
                f"micro_{RUN_LABELS[0]}",
                f"micro_{RUN_LABELS[1]}",
                "micro_delta",
            ]
        ]
        .sort_values("mean_delta")
        .round(4)
    )

    plot_field_delta(field_delta, len(comparison["names"]))

## Quoting these numbers

A short decision list, because the tables above deliberately disagree with each other:

| You want to say | Use | From |
|---|---|---|
| "the run scored X" (matching the pipeline log) | `Avg F1 (mean)` | execution summary |
| "a typical document scores X on field F" | `mean_f1` | macro table |
| "we correctly extracted X% of all transactions" | `micro_f1` | micro table |
| "field F is unreliable" | `min_f1` + the worst-documents table | macro table |
| "the failure mode is truncation / hallucination" | `micro_recall` vs `micro_precision` | micro table |
| "degradation cost us X on field F" | `mean_delta` + `sd_delta` | per-field delta table |

Rules that follow from the traps above:

1. **Never quote a headline F1 without its `n`.** Both the document count and, for micro numbers,
   `n_items` change what the figure means.
2. **Never reach for `Avg F1 (median)` as the robust or conservative option.** It discards the
   weakest fields by construction, and on this file it is also the noisier of the two. If you want
   robustness across documents, use the `median` column of the execution summary — that one is a
   genuine corpus-level median.
3. **Never compare headlines across runs with different document-type mixes.** Compare the
   per-type tables, or a change in the mix will read as a change in quality.
4. **State the floor when quoting outside the team.** "0.887 on a scale whose floor is 0.400" is a
   different claim from "0.887 out of 1.0", and only the first is true.
5. **Do not report a corpus-level improvement without checking the worst documents.** The loss is
   concentrated, so a headline can move because one long statement got better while nothing else
   changed — or stay flat while a genuine fix is masked by one regression.
6. **Never quote a clean-vs-degraded difference as a difference of two headlines.** Use the paired
   delta, which is a different and smaller-variance quantity, and never quote it corpus-wide when
   only one document type was degraded.